# 33 · Workflows: prompt chaining and routing
Level L3 · Part 4 · core · ~35 min · builds on 19, 20

## Goal
Build a two-step prompt chain (draft a summary, then check it) with a code gate between the steps, and
a router that sends each of Scout's requests to a summarise, calculate, search or fallback path.

## The idea
In the agent loop (lesson 26) the model decides the next step. A **workflow** is different: your code
fixes the steps and the model fills in each one. **Prompt chaining** splits a task into a fixed sequence
of model calls, each reading the previous output. A **gate** is a plain code check between two steps: if
the draft breaks a rule, the chain stops before it spends another call. **Routing** classifies a request
once, then sends it to the handler built for that kind of request. You know every path in advance, so
both are easy to test.

Analogy: a factory line with an inspector between stations, and a sorting desk at the door.

## Picture
```mermaid
flowchart LR
    Q["request"] --> R{"router<br/>(1 model call)"}
    R -- summarise --> D["draft<br/>(model)"] --> G{"gate<br/>(code)"}
    G -- pass --> C["check<br/>(model)"] --> A["answer"]
    G -- fail --> X["stop, report why"]
    R -- calculate --> K["calculator"] --> A
    R -- search --> S["search_notes"] --> A
    R -- other --> F["fallback reply"]
```

## Step by step
### 1. Start the recorder and set up
The recorder first, as in lesson 20. Without a key the Gemini calls replay from
`../data/llm_cassettes/33_workflows-chaining-routing.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("33_workflows-chaining-routing")

LLM mode: replay (33_workflows-chaining-routing.yaml)


Setup from earlier lessons, unchanged: the notes, client and model id (lesson 07), and `search_notes`
and the safe `calculator` (lesson 26).

In [2]:
import ast, operator, os, re
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
print(len(NOTES), "notes | model:", MODEL)

6 notes | model: gemini-3.5-flash-lite


The two local tools, copied from lesson 26.

In [3]:
def search_notes(query: str) -> list[dict]:
    """Return the notes whose text contains `query` (ignoring case)."""
    q = query.lower()
    return [note for note in NOTES if q in note["text"].lower()]

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def calculator(expression: str) -> float:
    """Evaluate +, -, *, / on numbers. Nothing else is allowed."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](ev(node.left), ev(node.right))
        raise ValueError(f"not allowed in a calculation: {ast.dump(node)[:40]}")
    return ev(ast.parse(expression, mode="eval").body)

print(len(search_notes("Mars")), "Mars notes |", calculator("24*60+39"))

3 Mars notes | 1479


### 2. Chain step 1: draft a summary
The first step turns a list of notes into a short summary that cites note ids, like `[3]`. The prompt
follows lesson 19: data first in XML tags, then exact instructions with the reason. One small `ask`
helper sends a prompt and returns the reply text.

In [4]:
def ask(prompt: str, schema=None):
    config = types.GenerateContentConfig(temperature=0, max_output_tokens=200, automatic_function_calling=NO_AFC)
    if schema is not None:
        config.response_mime_type, config.response_schema = "application/json", schema
    resp = client.models.generate_content(model=MODEL, contents=prompt, config=config)
    return resp.parsed if schema is not None else (resp.text or "").strip()

def notes_xml(notes: list[dict]) -> str:
    return "\n".join(f'<note id="{n["id"]}">{n["text"]}</note>' for n in notes)

def draft_prompt(notes: list[dict]) -> str:
    return (f"<notes>\n{notes_xml(notes)}\n</notes>\n\n<instructions>\nSummarise the notes above in at "
            "most 40 words, as plain prose. After each fact, cite its note id in square brackets, like [3]. "
            "Scout links each [id] to the note, so use only ids from the notes above.\n</instructions>")

Run step 1 on the three Mars notes.

In [5]:
mars_notes = search_notes("Mars")
draft = ask(draft_prompt(mars_notes))
print(draft)

Mars has two moons, Phobos and Deimos [1]. A Martian day lasts about 24 hours and 39 minutes [3]. Additionally, the rover Perseverance successfully landed on the planet in February 2021 [5].


### 3. The gate: plain code between the steps
Before we pay for step 2, code checks what code can check: the length, that there is at least one
citation, and that every cited id is one of the notes we sent. It returns the broken rules, like the
checkers in lessons 19 and 20, so an empty list means "pass".

In [6]:
def gate(draft: str, notes: list[dict]) -> list[str]:
    problems = []
    words = len(draft.split())
    if words > 40:
        problems.append(f"{words} words (max 40)")
    cited = {int(i) for i in re.findall(r"\[(\d+)\]", draft)}
    if not cited:
        problems.append("no [id] citations")
    if cited - {n["id"] for n in notes}:
        problems.append(f"cites unknown notes {sorted(cited - {n['id'] for n in notes})}")
    return problems

print("model draft:", gate(draft, mars_notes) or "pass")

model draft: pass


Now two drafts we wrote by hand, to see the gate fail. The first cites note 4, which is about Jupiter and
was never sent. The second has no citations at all.

In [7]:
bad_drafts = ["Mars has two moons [1] and is the largest planet [4].",
              "Mars has two moons and a day of about 24 hours and 39 minutes."]
for d in bad_drafts:
    print(gate(d, mars_notes))

['cites unknown notes [4]']
['no [id] citations']


### 4. Chain step 2: check the draft against the notes
The gate cannot tell whether a sentence is *true to the notes*. That needs a model. Step 2 gets the notes
and the draft and returns a typed verdict, using a Pydantic class as the schema (lesson 20), because code
reads this step's output.

In [8]:
from pydantic import BaseModel, Field

class Verdict(BaseModel):
    supported: bool = Field(description="true only if every fact in the draft is stated in the notes")
    unsupported: list[str] = Field(description="facts in the draft that the notes do not state; empty if none")

def check_prompt(draft: str, notes: list[dict]) -> str:
    return (f"<notes>\n{notes_xml(notes)}\n</notes>\n\n<draft>{draft}</draft>\n\n<instructions>\nCheck the "
            "draft against the notes. List any fact in the draft that the notes do not state.\n</instructions>")

print(Verdict.model_json_schema()["required"])

['supported', 'unsupported']


### 5. The whole chain
`run_chain` wires the steps together: draft, gate, check. If the gate fails, it stops and says why;
step 2 never runs. It also counts model calls, so we can see what each path costs.

In [9]:
def run_chain(notes: list[dict], draft_step=None) -> dict:
    calls = 0
    if draft_step is None:
        draft, calls = ask(draft_prompt(notes)), 1
    else:
        draft = draft_step(notes)                      # lets us plug in a hand-written draft
    problems = gate(draft, notes)
    if problems:
        return {"draft": draft, "ok": False, "why": problems, "calls": calls}
    verdict = ask(check_prompt(draft, notes), schema=Verdict)
    return {"draft": draft, "ok": verdict.supported, "why": verdict.unsupported, "calls": calls + 1}

First with a hand-written bad draft: the gate stops it, and no model call is made.

In [10]:
stopped = run_chain(mars_notes, draft_step=lambda notes: bad_drafts[0])
print(stopped)

{'draft': 'Mars has two moons [1] and is the largest planet [4].', 'ok': False, 'why': ['cites unknown notes [4]'], 'calls': 0}


Now the real chain on the Mars notes: one call to draft, one to check.

In [11]:
mars = run_chain(mars_notes)
print(mars)

{'draft': 'Mars features two small moons, Phobos and Deimos [1]. A Martian day lasts approximately 24 hours and 39 minutes [3]. Additionally, the Perseverance rover successfully landed on the planet in February 2021 [5].', 'ok': True, 'why': [], 'calls': 2}


### 6. Routing: a keyword router first
Scout gets different kinds of requests. Each needs a different handler, so we first decide which kind a
request is. The cheapest router is plain code: look for keywords. Here are six requests and a keyword
router; `None` means "no rule matched".

In [12]:
REQUESTS = [
    "Summarise my notes about Mars",
    "How many minutes is 24 hours and 39 minutes?",
    "Which note mentions Saturn?",
    "Give me a short overview of what I wrote on Jupiter",
    "What is 1.5 million km in metres?",
    "Book me a flight to Mars",
]

def keyword_route(request: str) -> str | None:
    r = request.lower()
    if "summar" in r:
        return "summarise"
    if "how many" in r or re.search(r"\d", r):
        return "calculate"
    if "which note" in r or "find" in r:
        return "search"
    return None

for req in REQUESTS:
    print(f"{keyword_route(req)!s:10} <- {req}")

summarise  <- Summarise my notes about Mars
calculate  <- How many minutes is 24 hours and 39 minutes?
search     <- Which note mentions Saturn?
None       <- Give me a short overview of what I wrote on Jupiter
calculate  <- What is 1.5 million km in metres?
None       <- Book me a flight to Mars


Five of six are handled correctly. The one real miss is "Give me a short overview…": it means
*summarise* but matched no rule, so it got `None`. The flight request also got `None`, which is right
only because we want it to fall through to a fallback. More rules help for a while, then they fight
each other. Keyword routers are fine when requests use a fixed vocabulary; for free text, use a model.

### 7. A model router with a fixed set of routes
The router is one model call that returns a route **and** the argument its handler needs. A `Literal`
type turns the routes into an enum in the schema (lesson 20), so the model cannot invent a sixth route.
`other` is the fallback for anything Scout should not handle.

In [13]:
from typing import Literal

class Route(BaseModel):
    route: Literal["summarise", "calculate", "search", "other"]
    argument: str = Field(description="summarise/search: one topic word, e.g. 'Mars'. "
                          "calculate: an arithmetic expression with numbers and + - * / only. other: ''")

def route_prompt(request: str) -> str:
    return (f"<request>{request}</request>\n\n<instructions>\nYou route requests for Scout, a research "
            "assistant over the user's notes. Pick the route and its argument.\n</instructions>")

print(Route.model_json_schema()["properties"]["route"])

{'enum': ['summarise', 'calculate', 'search', 'other'], 'title': 'Route', 'type': 'string'}


Route all six requests: six model calls.

In [14]:
routes = [ask(route_prompt(req), schema=Route) for req in REQUESTS]
for req, r in zip(REQUESTS, routes):
    print(f"{r.route:10} {r.argument!r:22} <- {req}")

summarise  'Mars'                 <- Summarise my notes about Mars
calculate  '24 * 60 + 39'         <- How many minutes is 24 hours and 39 minutes?
search     'Saturn'               <- Which note mentions Saturn?
summarise  'Jupiter'              <- Give me a short overview of what I wrote on Jupiter
calculate  '1500000 * 1000'       <- What is 1.5 million km in metres?
other      ''                     <- Book me a flight to Mars


### 8. Handlers: one path per route
Each route gets a small handler. Only `summarise` uses more model calls (the chain); `calculate` and
`search` are plain code. A table maps route names to handlers, the same idea as the tool registry.

In [15]:
def handle_summarise(arg: str) -> str:
    notes = search_notes(arg)
    if not notes:
        return f"No notes about {arg!r}."
    result = run_chain(notes)
    return result["draft"] if result["ok"] else f"Draft rejected: {result['why']}"

def handle_calculate(arg: str) -> str:
    return f"{arg} = {calculator(arg):,.0f}"

def handle_search(arg: str) -> str:
    return "; ".join(f"[{n['id']}] {n['text']}" for n in search_notes(arg)) or "No match."

def handle_other(arg: str) -> str:
    return "Sorry, Scout only summarises, searches and calculates over your notes."

HANDLERS = {"summarise": handle_summarise, "calculate": handle_calculate,
            "search": handle_search, "other": handle_other}

Dispatch each routed request. Only the Jupiter request reaches `summarise`, so this cell makes the
chain's two model calls once more.

In [16]:
for req, r in zip(REQUESTS[1:], routes[1:]):          # request 0 (Mars) already ran through the chain in step 5
    print(f"{r.route:10} -> {HANDLERS[r.route](r.argument)}")

calculate  -> 24 * 60 + 39 = 1,479
search     -> [6] Saturn's rings are mostly made of ice.
summarise  -> Jupiter is the largest planet in the solar system [4].
calculate  -> 1500000 * 1000 = 1,500,000,000
other      -> Sorry, Scout only summarises, searches and calculates over your notes.


Every request got an answer from the path built for it, and the flight request fell through to a polite
refusal instead of a made-up booking.

## At work
- **Simplest thing first.** Anthropic's "Building effective agents": start with one good call, use a
  workflow when the task splits into fixed steps, and an agent only when it must pick its own steps;
  agents trade latency and cost for flexibility. → lesson 34
- **Route easy requests to cheap models.** A router can be a cheap small-model call that picks the
  model or handler for the real work, so you pay for the big model only on hard requests (model
  routing, small model first, is a standard cost control). → lesson 54
- **Gates are tests.** Gate rules also run in the test suite and evals, so a prompt change that breaks
  the format fails in CI, not in production. → lesson 48

## What just happened
- A **workflow** fixes the steps in code; the model only fills each step in.
- The **chain** drafted a cited summary, then a second call checked it against the notes, returning a
  typed `Verdict`.
- The **gate** between them is plain code. It stopped a hand-written bad draft before any model call.
- A **keyword router** handled fixed phrases but missed "overview"; any request with no matching rule
  returns `None`, so it needs a fallback.
- A **model router** returned a route from a fixed enum plus its argument in one call; a handler table
  then ran exactly one path per request, with `other` as the fallback.

## Common mistakes
**1. A router with no fallback.** The keyword router returns `None` when no rule matches, and the handler
table has no `None` key, so the request crashes Scout.

In [17]:
for req in REQUESTS:
    handler = HANDLERS[keyword_route(req)]
    print(f"{handler.__name__:18} <- {req}")

handle_summarise   <- Summarise my notes about Mars
handle_calculate   <- How many minutes is 24 hours and 39 minutes?
handle_search      <- Which note mentions Saturn?


KeyError: None

Fix: always have a fallback route, and send anything unmatched there. Scout no longer crashes, though
the Jupiter overview still lands in `other` (the flight request landing there is correct): the reason
step 7 routes with a model.

In [18]:
for req in REQUESTS:
    handler = HANDLERS[keyword_route(req) or "other"]
    print(f"{handler.__name__:18} <- {req}")

handle_summarise   <- Summarise my notes about Mars
handle_calculate   <- How many minutes is 24 hours and 39 minutes?
handle_search      <- Which note mentions Saturn?
handle_other       <- Give me a short overview of what I wrote on Jupiter
handle_calculate   <- What is 1.5 million km in metres?
handle_other       <- Book me a flight to Mars


**2. Parsing a step's output that has no format.** Step 1 was asked for plain prose. Code that later
assumes it returned JSON breaks on the first real reply.

In [19]:
import json
summary = json.loads(mars["draft"])["summary"]

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Fix: decide each step's output format on purpose. Text that only a person or the next prompt reads can
be prose; output that code reads, like the check step and the router, gets a schema.

In [20]:
print("step 1 (prose):", type(mars["draft"]).__name__, "| router (schema):", type(routes[0]).__name__, routes[0].route)

step 1 (prose): str | router (schema): Route summarise


## Try it
Add a new gate rule: every sentence needs at least one `[id]`. Write `gate_strict(draft, notes)`, which
returns `gate`'s problems plus this one, and test it on `bad_drafts`, on
`"Mars has two moons [1]. It is red."` and on `mars["draft"]`.

In [21]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [22]:
llm_replay.stop()

## Key terms
- **workflow** — an LLM system whose steps are fixed in code; the model fills in each step but does not choose the next one.
- **prompt chaining** — splitting a task into a fixed sequence of model calls, each one reading the previous call's output.
- **gate** — a plain code check between two chain steps that stops the chain when an output breaks a rule.
- **routing** — classifying a request once and sending it to the handler built for that kind of request.
- **fallback route** — the route for requests no other route fits, so an unmatched request gets a safe reply instead of an error.